Ingestão

In [ ]:
import sys
from pathlib import Path

AGENTS_DIR = Path.cwd().parent / "agents"
sys.path.append(str(AGENTS_DIR))

from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

In [ ]:
documents = documents_llm

In [ ]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

Geração do dataset

In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [ ]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
genai_client = genai.Client()

MODEL = "gemini-3.5-flash-lite"

In [ ]:
import json

user_prompt = json.dumps(doc)

No Gemini, as instruções vão em `system_instruction` e o formato da saída em `response_schema`, que aceita o próprio modelo Pydantic. O objeto já validado fica em `response.parsed` (equivalente ao `responses.parse(text_format=...)` / `output_parsed` da OpenAI).

In [ ]:
config = types.GenerateContentConfig(
    system_instruction=data_gen_instructions,
    response_mime_type="application/json",
    response_schema=Questions,
)

In [ ]:
response = genai_client.models.generate_content(
    model=MODEL,
    contents=user_prompt,
    config=config,
)

In [ ]:
result = response.parsed

print(result)

In [ ]:
print(result.questions)

Utils de avaliação

In [ ]:
from evaluation_utils import llm_structured

In [ ]:
result, usage = llm_structured(
    genai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
    model=MODEL,
)

print(result.questions)

Rastreio de custo

O Gemini devolve o uso em `response.usage_metadata` (`prompt_token_count`, `candidates_token_count`, `thoughts_token_count`). O `evaluation_utils` normaliza isso em `Usage(input_tokens, output_tokens)`, contando os tokens de raciocínio como saída, já que são cobrados assim.

In [ ]:
response.usage_metadata

In [ ]:
usage.input_tokens, usage.output_tokens

In [ ]:
from evaluation_utils import calc_price

In [ ]:
cost = calc_price(usage)

cost

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

Avaliação em batch

In [ ]:
from evaluation_utils import llm_structured_retry

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        genai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model=MODEL,
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

Processamento paralelo

No free tier do Gemini o limite de requisições por minuto é baixo: com muitos workers aparecem erros 429/503. O `llm_structured_retry` faz backoff exponencial, mas é melhor manter poucos workers.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)